# Module 2.1 — Convolutional Neural Networks

CNNs are the backbone of modern computer vision. Instead of flattening an image into a vector (as our MLP did), a CNN preserves spatial structure by sliding small filters across the image.

**What you'll learn:**
- How the convolution operation works: kernels, stride, padding
- Pooling and the receptive field
- LeNet (1998) → AlexNet (2012): the key architectural ideas
- Build and train a CNN on MNIST — compare it directly to the MLP from 1.3
- See where a simple CNN starts to struggle on CIFAR-10

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

print(f'PyTorch version: {torch.__version__}')
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. The Convolution Operation

### Why not just use an MLP?

Your MLP from 1.3 flattened each 28×28 image into a 784-element vector. That works, but it throws away every spatial relationship — a pixel's value is treated the same whether it's next to a related pixel or on the opposite side of the image.

A CNN instead slides a small **filter** (also called a **kernel**) across the image. At each position, it computes a dot product between the kernel weights and the image patch underneath. The result is a single number; sliding the kernel across the whole image produces a 2-D **feature map**.

### The key insight: weight sharing

The same kernel is applied at every position. This means:
- **Far fewer parameters** — a 5×5 kernel has 25 weights regardless of image size
- **Translation equivariance** — if a feature (say, a vertical edge) appears anywhere in the image, the same filter detects it

### Multiple filters → multiple feature maps

A conv layer uses many filters (e.g. 32), each learning to detect a different feature. The output is a stack of feature maps, one per filter.

**Output size formula** for input W×H, kernel K, padding P, stride S:
```
out_width  = floor((W - K + 2P) / S) + 1
out_height = floor((H - K + 2P) / S) + 1
```

In [ ]:
# Load one MNIST image (no normalization — easier to visualize raw pixels)
raw_transform = transforms.ToTensor()
mnist_raw = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=raw_transform)
img, label = mnist_raw[0]  # [1, 28, 28]
print(f'Image shape: {img.shape},  label: {label}')

# Three hand-crafted kernels — each detects a different feature
kernels = {
    'horizontal edge': torch.tensor([[-1., -1., -1.], [0., 0., 0.], [1., 1., 1.]]),
    'vertical edge':   torch.tensor([[-1.,  0.,  1.], [-1., 0., 1.], [-1., 0., 1.]]),
    'blur':            torch.ones(3, 3) / 9.0,
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
axes[0].imshow(img.squeeze(), cmap='gray')
axes[0].set_title(f'Original (label={label})')
axes[0].axis('off')

for ax, (name, kernel) in zip(axes[1:], kernels.items()):
    k = kernel.unsqueeze(0).unsqueeze(0)          # [1, 1, 3, 3] — shape conv2d expects
    out = F.conv2d(img.unsqueeze(0), k, padding=1) # padding=1 keeps spatial size
    ax.imshow(out.squeeze().numpy(), cmap='gray')
    ax.set_title(name)
    ax.axis('off')

plt.suptitle('Hand-crafted kernels — each detects a different feature')
plt.tight_layout()
plt.show()

## 3. Stride and Padding

**Padding** adds zeros around the border of the input:
- `padding=0` — the kernel can't reach the edges, so the output shrinks by `kernel_size - 1` in each dimension
- `padding=1` with a 3×3 kernel — output stays the same size as the input ("same" padding)

**Stride** controls how far the kernel moves between applications:
- `stride=1` (default) — kernel moves one pixel at a time
- `stride=2` — kernel skips every other position, roughly halving the spatial dimensions

Strided convolutions are an alternative to pooling for downsampling — modern architectures like ResNet often use strided convs instead of max pool.

In [ ]:
x = torch.randn(1, 1, 28, 28)

configs = [
    ('kernel=3, stride=1, pad=0', nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=0)),
    ('kernel=3, stride=1, pad=1', nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=1)),
    ('kernel=3, stride=2, pad=0', nn.Conv2d(1, 1, kernel_size=3, stride=2, padding=0)),
    ('kernel=5, stride=1, pad=2', nn.Conv2d(1, 1, kernel_size=5, stride=1, padding=2)),
]

print(f'Input size:  {list(x.shape)}')
print()
for name, conv in configs:
    with torch.no_grad():
        out = conv(x)
    print(f'{name:35s}  ->  output: {list(out.shape)}')

## 4. Pooling and the Receptive Field

### Pooling

After a conv layer we often apply a **pooling** layer to reduce spatial dimensions:

- **Max pooling** — takes the maximum value in each window. Says "did this feature appear anywhere in this region?" Most common in modern networks.
- **Average pooling** — takes the mean. Used in LeNet and for global pooling at network heads.

A 2×2 max pool with stride 2 halves the height and width (quarters the spatial size), with no learnable parameters.

### Receptive Field

The **receptive field** of a neuron is the region of the original input that influences its value. This grows with depth:

| Depth | Operation | Receptive field |
|---|---|---|
| 1 | 3×3 conv | 3×3 |
| 2 | + 3×3 conv | 5×5 |
| 3 | + 3×3 conv | 7×7 |
| — | one 7×7 conv | 7×7 |

Three stacked 3×3 layers match the receptive field of one 7×7 layer, but use `3×9 = 27` weights vs `49` — more efficient, and with two extra non-linearities.

In [ ]:
img_tensor = img.unsqueeze(0)  # [1, 1, 28, 28]

pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
pool4 = nn.MaxPool2d(kernel_size=4, stride=4)

with torch.no_grad():
    pooled2 = pool2(img_tensor)  # [1, 1, 14, 14]
    pooled4 = pool4(img_tensor)  # [1, 1, 7, 7]

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, (title, t) in zip(axes, [
    ('Original 28x28',    img.squeeze()),
    ('MaxPool(2) -> 14x14', pooled2.squeeze()),
    ('MaxPool(4) -> 7x7',   pooled4.squeeze()),
]):
    ax.imshow(t.numpy(), cmap='gray', interpolation='nearest')
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. LeNet-5 Architecture

LeNet-5 (LeCun et al., 1998) was trained on MNIST and is the ancestor of all modern CNNs. The design decisions it introduced — convolutional feature extraction followed by fully-connected classification — are still used today.

```
Input        [1 x 28 x 28]
   |
Conv(1->6, k=5, pad=2) + ReLU       [6 x 28 x 28]
   |
MaxPool(2)                           [6 x 14 x 14]
   |
Conv(6->16, k=5) + ReLU             [16 x 10 x 10]
   |
MaxPool(2)                           [16 x 5 x 5]
   |
Flatten                              [400]
   |
Linear(400->120) + ReLU             [120]
   |
Linear(120->84) + ReLU              [84]
   |
Linear(84->10)                       [10]
```

**Total parameters: ~62,000** — roughly 4× fewer than the 235K MLP from 1.3, yet more accurate because the conv layers exploit spatial structure.

Note: the original paper used average pooling and tanh activations. We use max pooling and ReLU, which is standard in modern practice.

In [ ]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5, padding=2),  # [6, 28, 28]
            nn.ReLU(),
            nn.MaxPool2d(2),                            # [6, 14, 14]
            nn.Conv2d(6, 16, kernel_size=5),            # [16, 10, 10]
            nn.ReLU(),
            nn.MaxPool2d(2),                            # [16, 5, 5]
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 5 * 5, 120),
            nn.ReLU(),
            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Linear(84, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = LeNet().to(device)
print(model)

lenet_params = sum(p.numel() for p in model.parameters())
print(f'\nLeNet parameters:  {lenet_params:,}')
print(f'MLP parameters:    235,146')
print(f'Reduction:         {235146 / lenet_params:.1f}x fewer')

## 6. Load MNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_train = torchvision.datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_set   = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_set, val_set = random_split(full_train, [50000, 10000])

train_loader = DataLoader(train_set, batch_size=64, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=64, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=64, shuffle=False, num_workers=0)

print(f'Train: {len(train_set):,}  |  Val: {len(val_set):,}  |  Test: {len(test_set):,}')

## 7. Train LeNet

Same training loop as 1.3 — the CNN is just another `nn.Module`.

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct = 0.0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0.0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        total_loss += criterion(logits, labels).item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 10
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    print(f'Epoch {epoch:02d} | Train loss: {train_loss:.4f}, acc: {train_acc:.3f} | '
          f'Val loss: {val_loss:.4f}, acc: {val_acc:.3f}')

In [ ]:
epochs = range(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history['train_loss'], label='Train')
ax1.plot(epochs, history['val_loss'],   label='Validation')
ax1.set_title('LeNet Loss')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(epochs, history['train_acc'], label='Train')
ax2.plot(epochs, history['val_acc'],   label='Validation')
ax2.set_title('LeNet Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.show()

test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f'Test accuracy: {test_acc:.4f}  ({test_acc * 100:.2f}%)')
print('MLP from 1.3 achieved 96.52% — did the CNN do better with fewer parameters?')

## 8. Visualise Learned Filters

After training, the first conv layer's 6 filters have each learned to detect a different low-level feature. We can look at the raw filter weights and also see what each filter activates on for a real image.

In [ ]:
# The 6 learned 5x5 filters from conv1
first_conv = model.features[0]
weights = first_conv.weight.data.cpu()  # [6, 1, 5, 5]
vmax = weights.abs().max().item()

fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(weights[i, 0].numpy(), cmap='RdBu', vmin=-vmax, vmax=vmax)
    ax.set_title(f'Filter {i + 1}')
    ax.axis('off')
plt.suptitle('Learned conv1 filters (5x5 each)')
plt.tight_layout()
plt.show()

In [ ]:
# What each filter responds to on a real image
img_norm, lbl = full_train[0]
x_in = img_norm.unsqueeze(0).to(device)

with torch.no_grad():
    after_conv1 = F.relu(model.features[0](x_in))  # [1, 6, 28, 28]

fig, axes = plt.subplots(1, 7, figsize=(14, 2))
axes[0].imshow(img_norm.squeeze(), cmap='gray')
axes[0].set_title(f'Input (label={lbl})')
axes[0].axis('off')

for i in range(6):
    axes[i + 1].imshow(after_conv1[0, i].cpu().numpy(), cmap='viridis')
    axes[i + 1].set_title(f'Map {i + 1}')
    axes[i + 1].axis('off')

plt.suptitle('Feature maps after conv1 + ReLU')
plt.tight_layout()
plt.show()

## 9. AlexNet — From 1998 to 2012

In 2012, AlexNet won ImageNet (1.2M images, 1,000 classes) by a ~10 percentage-point margin over the previous best. It used the same fundamental idea as LeNet but made four key changes:

| Change | Why it mattered |
|---|---|
| **ReLU instead of tanh** | Trains ~6× faster; doesn't saturate for large activations |
| **Dropout** | Reduces overfitting without shrinking the model |
| **Data augmentation** | Random crops + horizontal flips — effectively more training data |
| **Two GPUs** | Enabled a much larger model (60M parameters) |

**Architecture overview** (for 224×224 RGB images):

```
Input        [3 x 224 x 224]
Conv(3->96, k=11, s=4) + ReLU + MaxPool    [96 x 27 x 27]
Conv(96->256, k=5, pad=2) + ReLU + MaxPool [256 x 13 x 13]
Conv(256->384, k=3, pad=1) + ReLU          [384 x 13 x 13]
Conv(384->384, k=3, pad=1) + ReLU          [384 x 13 x 13]
Conv(384->256, k=3, pad=1) + ReLU + MaxPool[256 x 6 x 6]
Flatten -> 9216
Linear(9216->4096) + ReLU + Dropout(0.5)
Linear(4096->4096) + ReLU + Dropout(0.5)
Linear(4096->1000)
```

The progression LeNet → AlexNet → modern networks (VGG, ResNet) is the same basic structure, but deeper, with better regularisation and training techniques. Module 2.2 covers batch norm, residual connections, and transfer learning.

In [ ]:
# AlexNet definition — for reference, not trained here (needs ImageNet-scale data)
class AlexNet(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=11, stride=4), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(96, 256, kernel_size=5, padding=2), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(256, 384, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(384, 384, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5), nn.Linear(256 * 6 * 6, 4096), nn.ReLU(),
            nn.Dropout(0.5), nn.Linear(4096, 4096), nn.ReLU(),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


alexnet = AlexNet()
alexnet_params = sum(p.numel() for p in alexnet.parameters())
print(f'AlexNet parameters: {alexnet_params:,}')
print(f'LeNet parameters:   {lenet_params:,}')
print(f'Ratio: {alexnet_params / lenet_params:.0f}x more parameters')

## 10. CIFAR-10

MNIST is nearly solved — even a simple MLP gets >96%. Let's try something harder: **CIFAR-10**.

- 60,000 **colour** images (3-channel RGB), 32×32 pixels
- 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck
- Much higher intra-class variation than handwritten digits

We'll build a 3-block CNN to handle the extra complexity. Notice the first layer changes from `Conv2d(1, ...)` to `Conv2d(3, ...)` — three input channels for R, G, B.

In [ ]:
cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

cifar_train_full = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=cifar_transform)
cifar_test       = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=cifar_transform)
cifar_train, cifar_val = random_split(cifar_train_full, [45000, 5000])

cifar_train_loader = DataLoader(cifar_train, batch_size=64, shuffle=True,  num_workers=0)
cifar_val_loader   = DataLoader(cifar_val,   batch_size=64, shuffle=False, num_workers=0)
cifar_test_loader  = DataLoader(cifar_test,  batch_size=64, shuffle=False, num_workers=0)

classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
print(f'Train: {len(cifar_train):,}  |  Val: {len(cifar_val):,}  |  Test: {len(cifar_test):,}')

images, labels = next(iter(cifar_train_loader))
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    img_show = images[i].permute(1, 2, 0) * 0.5 + 0.5  # unnormalize to [0, 1]
    ax.imshow(img_show.numpy())
    ax.set_title(classes[labels[i].item()], fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
class CifarCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: [3, 32, 32] -> [32, 16, 16]
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 2: [32, 16, 16] -> [64, 8, 8]
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 3: [64, 8, 8] -> [128, 4, 4]
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


cifar_model = CifarCNN().to(device)
total_cifar = sum(p.numel() for p in cifar_model.parameters())
print(cifar_model)
print(f'\nTotal parameters: {total_cifar:,}')

In [ ]:
cifar_criterion = nn.CrossEntropyLoss()
cifar_optimizer = optim.Adam(cifar_model.parameters(), lr=1e-3)

CIFAR_EPOCHS = 15
cifar_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, CIFAR_EPOCHS + 1):
    train_loss, train_acc = train_epoch(cifar_model, cifar_train_loader, cifar_criterion, cifar_optimizer)
    val_loss,   val_acc   = evaluate(cifar_model, cifar_val_loader, cifar_criterion)
    cifar_history['train_loss'].append(train_loss)
    cifar_history['val_loss'].append(val_loss)
    cifar_history['train_acc'].append(train_acc)
    cifar_history['val_acc'].append(val_acc)
    print(f'Epoch {epoch:02d} | Train loss: {train_loss:.4f}, acc: {train_acc:.3f} | '
          f'Val loss: {val_loss:.4f}, acc: {val_acc:.3f}')

cifar_test_loss, cifar_test_acc = evaluate(cifar_model, cifar_test_loader, cifar_criterion)
print(f'\nTest accuracy: {cifar_test_acc:.4f}  ({cifar_test_acc * 100:.2f}%)')

In [ ]:
epochs_c = range(1, CIFAR_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_c, cifar_history['train_loss'], label='Train')
ax1.plot(epochs_c, cifar_history['val_loss'],   label='Validation')
ax1.set_title('CIFAR-10 Loss')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(epochs_c, cifar_history['train_acc'], label='Train')
ax2.plot(epochs_c, cifar_history['val_acc'],   label='Validation')
ax2.set_title('CIFAR-10 Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Per-class accuracy — some classes are much harder than others
@torch.no_grad()
def per_class_accuracy(model, loader, num_classes=10):
    model.eval()
    correct = torch.zeros(num_classes)
    total   = torch.zeros(num_classes)
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        for c in range(num_classes):
            mask = labels == c
            correct[c] += (preds[mask] == c).sum().item()
            total[c]   += mask.sum().item()
    return (correct / total).numpy()


accs = per_class_accuracy(cifar_model, cifar_test_loader)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(classes, accs)
ax.set_ylim(0, 1)
ax.set_ylabel('Accuracy')
ax.set_title('Per-class accuracy on CIFAR-10 test set')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{acc:.2f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()
print('cat/dog tend to be confused; ship/plane tend to be easier')

## Summary

| Concept | Key takeaway |
|---|---|
| Convolution | Slides a kernel across the image; same weights everywhere (weight sharing) |
| Stride / padding | Stride controls downsampling speed; padding controls border handling |
| Max pooling | Reduces spatial size; adds translation invariance |
| Receptive field | Grows with depth; stacked 3×3 convs beat one large kernel on efficiency |
| LeNet (1998) | 2 conv + 3 FC layers; ~62K params; beats the MLP from 1.3 with 4× fewer weights |
| AlexNet (2012) | Same idea but deeper + ReLU + dropout; won ImageNet, kickstarted deep learning |
| CIFAR-10 | Harder than MNIST; ~70-75% with a basic CNN — we need more tricks |

**What's holding the CIFAR CNN back?**

The gap between training and validation accuracy means the network is overfitting. There's no batch normalisation, no dropout, no data augmentation. Module 2.2 introduces exactly those techniques — plus residual connections and transfer learning — to push well above 90%.